In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install pillow torchvision

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dxby6ej2/unsloth_3e309fe6340643bf8ec4906b72c18524
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dxby6ej2/unsloth_3e309fe6340643bf8ec4906b72c18524
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.9 MB/s eta 0:00:00

In [2]:
import gc
import json
import os
import urllib.request
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torchvision import transforms
from PIL import Image

# ── Unsloth must be imported before transformers ──────────────────────────────
from unsloth import FastLanguageModel

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MAX_SEQ_LEN  = 2048
LORA_R       = 16      
LORA_ALPHA   = 32
LORA_DROPOUT = 0        
LOAD_IN_4BIT = True
BATCH_SIZE   = 2
ACCUMULATION = 4
EPOCHS       = 4
LR           = 1e-5    
DATA_DIR     = "/kaggle/working/coco"
MAX_SAMPLES  = 5000
SEQ_LEN      = 64      

# ─────────────────────────────────────────────────────────────────────────────
# DATA UTILS
# ─────────────────────────────────────────────────────────────────────────────
def download_coco(data_dir=DATA_DIR, max_samples=MAX_SAMPLES):
    os.makedirs(data_dir, exist_ok=True)
    images_dir = os.path.join(data_dir, "images")
    ann_dir = os.path.join(data_dir, "annotations")
    os.makedirs(images_dir, exist_ok=True)

    ann_zip = os.path.join(data_dir, "annotations.zip")
    captions_file = os.path.join(ann_dir, "captions_train2017.json")

    if not os.path.exists(captions_file):
        if not os.path.exists(ann_zip):
            print("Downloading COCO annotations...")
            urllib.request.urlretrieve(
                "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
                ann_zip,
            )
        print("Extracting annotations...")
        with zipfile.ZipFile(ann_zip, "r") as z:
            for member in z.namelist():
                if "captions_train2017" in member:
                    z.extract(member, data_dir)
        print("Annotations ready.")

    # Image Check
    existing = [f for f in os.listdir(images_dir) if f.endswith(".jpg")]
    if len(existing) < 10: # If empty or near empty
        print(f"Downloading sample images...")
        with open(captions_file, "r") as f:
            cap_data = json.load(f)
       
        for i, img_info in enumerate(cap_data["images"][:max_samples]):
            img_path = os.path.join(images_dir, img_info["file_name"])
            if not os.path.exists(img_path):
                try:
                    urllib.request.urlretrieve(
                        f"http://images.cocodataset.org/train2017/{img_info['file_name']}",
                        img_path,
                    )
                except: continue
            if (i + 1) % 500 == 0: print(f"  {i+1} images downloaded...")

    return images_dir, captions_file

class COCOCaptionDataset(Dataset):
    def __init__(self, images_dir, captions_file, tokenizer, hidden_dim):
        self.images_dir = images_dir
        self.tokenizer = tokenizer
        self.hidden_dim = hidden_dim

        with open(captions_file, "r") as f:
            coco = json.load(f)

        id_to_fname = {img["id"]: img["file_name"] for img in coco["images"]}
        self.samples = []
        for ann in coco["annotations"]:
            fname = id_to_fname.get(ann["image_id"])
            if fname:
                fpath = os.path.join(images_dir, fname)
                if os.path.exists(fpath):
                    self.samples.append((fpath, ann["caption"].strip()))
            if len(self.samples) >= MAX_SAMPLES:
                break

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, caption = self.samples[index]
        img_tensor = self.transform(Image.open(image_path).convert("RGB"))
        full_text = f"Describe this image: {caption}{self.tokenizer.eos_token}"

        enc = self.tokenizer(
            full_text,
            max_length=SEQ_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
       
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "image":          img_tensor,
            "target_velocities": torch.randn(SEQ_LEN, self.hidden_dim),
            "timestamp":      torch.randint(0, 1000, (1,)).float(),
        }

# ─────────────────────────────────────────────────────────────────────────────
# DIFFUSION HEAD
# ─────────────────────────────────────────────────────────────────────────────
class BinaryDiffusionHead(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, hidden_states, timestamps):
        B, S, H = hidden_states.shape
        t = timestamps.view(B, 1, 1).expand(B, S, 1)
        x = torch.cat([hidden_states, t], dim=-1)
        return self.net(x)

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train_model(model, diffusion_head, dataloader, device):
    optimizer = AdamW(
        list(filter(lambda p: p.requires_grad, model.parameters())) +
        list(diffusion_head.parameters()),
        lr=LR, eps=1e-4
    )
   
    scaler = torch.amp.GradScaler('cuda')
    model.train()
    diffusion_head.train()

    for epoch in range(EPOCHS):
        for step, batch in enumerate(dataloader):
            input_ids = batch["input_ids"].to(device)
            att_mask  = batch["attention_mask"].to(device)
            velocities = batch["target_velocities"].to(device)
            timestamps = batch["timestamp"].to(device)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=att_mask,
                                output_hidden_states=True, return_dict=True)
               
                # Causal Alignment for Accuracy
                logits = outputs.logits[:, :-1, :].contiguous()
                labels = input_ids[:, 1:].contiguous()
                mask   = att_mask[:, 1:].contiguous()
               
                loss_text = F.cross_entropy(
                    logits.float().view(-1, logits.size(-1)),
                    labels.where(mask.bool(), torch.tensor(-100, device=device)).view(-1),
                    ignore_index=-100
                )

                hidden_states = outputs.hidden_states[-1].float()
                pred_vel = diffusion_head(hidden_states, timestamps.float())
                loss_visual = F.mse_loss(pred_vel, velocities.float())

                loss = (0.1 * loss_text) + loss_visual

            if torch.isnan(loss):
                optimizer.zero_grad(); continue

            scaler.scale(loss / ACCUMULATION).backward()

            if (step + 1) % ACCUMULATION == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            if step % 20 == 0:
                preds = torch.argmax(logits, dim=-1)
                valid_mask = (labels != -100) & (mask == 1)
                acc = (preds[valid_mask] == labels[valid_mask]).float().mean() if valid_mask.any() else 0.0
                print(f"Ep {epoch} | Step {step} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    # 1. Prep Data
    img_dir, ann_file = download_coco()

    # 2. Prep Model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LEN,
        load_in_4bit = True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_ALPHA, lora_dropout=0,
    )
   
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
   
    # 3. Data Loader
    dataset = COCOCaptionDataset(img_dir, ann_file, tokenizer, model.config.hidden_size)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

    # 4. Train
    diffusion_head = BinaryDiffusionHead(model.config.hidden_size).to("cuda")
    train_model(model, diffusion_head, dataloader, "cuda")

if __name__ == "__main__":
    main()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Extracting annotations...
Annotations ready.
  500 images downloaded...
  1000 images downloaded...
  1500 images downloaded...
  2000 images downloaded...
  2500 images downloaded...
  3000 images downloaded...
  3500 images downloaded...
  4000 images downloaded...
  4500 images downloaded...
  5000 images downloaded...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.


Ep 0 | Step 0 | Loss: 7.7330 | Acc: 0.3448
Ep 0 | Step 20 | Loss: 6.7083 | Acc: 0.5152
Ep 0 | Step 40 | Loss: 2.4313 | Acc: 0.2759
Ep 0 | Step 60 | Loss: 2.3518 | Acc: 0.4074
Ep 0 | Step 80 | Loss: 1.6369 | Acc: 0.3438
Ep 0 | Step 100 | Loss: 1.7875 | Acc: 0.2333
Ep 0 | Step 120 | Loss: 1.6673 | Acc: 0.2692
Ep 0 | Step 140 | Loss: 1.5822 | Acc: 0.4194
Ep 0 | Step 160 | Loss: 1.5815 | Acc: 0.3871
Ep 0 | Step 180 | Loss: 1.4853 | Acc: 0.3704
Ep 0 | Step 200 | Loss: 1.5318 | Acc: 0.3438
Ep 0 | Step 220 | Loss: 1.5634 | Acc: 0.3667
Ep 0 | Step 240 | Loss: 1.4253 | Acc: 0.3750
Ep 0 | Step 260 | Loss: 1.5121 | Acc: 0.3929
Ep 0 | Step 280 | Loss: 1.4671 | Acc: 0.4839
Ep 0 | Step 300 | Loss: 1.4375 | Acc: 0.3793
Ep 0 | Step 320 | Loss: 1.5537 | Acc: 0.3000
Ep 0 | Step 340 | Loss: 1.3974 | Acc: 0.4000
Ep 0 | Step 360 | Loss: 1.3993 | Acc: 0.4444
Ep 0 | Step 380 | Loss: 1.4124 | Acc: 0.3939
Ep 0 | Step 400 | Loss: 1.3067 | Acc: 0.4688
Ep 0 | Step 420 | Loss: 1.3458 | Acc: 0.5625
Ep 0 | Step 440 

In [3]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-deps", "unsloth",
    "unsloth_zoo",
    "bitsandbytes",
    "peft",
    "trl",
    "accelerate",
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', 'unsloth', 'unsloth_zoo', 'bitsandbytes', 'peft', 'trl', 'accelerate'], returncode=0)